# 01 — Data Preparation

Loads the raw credit dataset, treats missing values, splits into Train / Test / Out-of-Time (OOT) samples, and fits Weight-of-Evidence (WoE) bins on the **training sample only** (to avoid leakage).

This notebook plays the role of the data pipeline a Model Development team would run before fitting a scorecard.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from data_processing import (load_raw_data, treat_missing_values,
    train_test_oot_split, fit_woe_all_features, apply_woe_all_features, RAW_FEATURES, TARGET)

## Load and inspect raw data

In [ ]:
df = load_raw_data('../data/raw/credit_data.csv')
print(df.shape)
df.head()

In [ ]:
df.isna().sum()

Missing values are present in `MonthlyIncome` (~5%) and `NumberOfDependents` (~3%), consistent with the real-world Give Me Some Credit dataset this synthetic data mirrors.

## Treat missing values

In [ ]:
df = treat_missing_values(df)
df[['MonthlyIncome_missing_flag', 'NumberOfDependents_missing_flag']].mean()

## Train / Test / Out-of-Time split

A real validator always insists on an **out-of-time (OOT)** holdout — not just a random split — because random splits can hide performance deterioration over time.

In [ ]:
train, test, oot = train_test_oot_split(df, oot_months=4)
print(f'Train: {len(train):,} | Test: {len(test):,} | OOT: {len(oot):,}')
print(f'Train default rate: {train[TARGET].mean():.4f}')
print(f'Test default rate:  {test[TARGET].mean():.4f}')
print(f'OOT default rate:   {oot[TARGET].mean():.4f}')

Note the OOT default rate is already higher than train/test — this is the macro drift later picked up in the Independent Validation Report (Finding 4).

## Fit WoE bins on TRAIN ONLY, apply to all three samples

In [ ]:
woe_maps = fit_woe_all_features(train, RAW_FEATURES, n_bins=5)
train_woe = apply_woe_all_features(train, woe_maps)
test_woe = apply_woe_all_features(test, woe_maps)
oot_woe = apply_woe_all_features(oot, woe_maps)
train_woe.filter(like='_woe').head()

In [ ]:
train_woe.to_csv('../data/processed/train_woe.csv', index=False)
test_woe.to_csv('../data/processed/test_woe.csv', index=False)
oot_woe.to_csv('../data/processed/oot_woe.csv', index=False)
print('Saved.')